# KnowRare in PyHealth: ICU mortality for rare conditions (MIMIC-IV)

<a href="https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/knowrare_mimic4_icu_mortality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs **KnowRare** (Zhu, Liu, Luo and Zhu, *Bridging data gaps of rare conditions in ICU: a multi-disease adaptation approach for clinical prediction*, npj Digital Medicine 2025, [arXiv:2507.06432](https://arxiv.org/abs/2507.06432); code: [JasonZuu/KnowRare](https://github.com/JasonZuu/KnowRare)) with PyHealth.

KnowRare predicts outcomes for ICU patients whose primary condition is rare. It pre-trains a time-series encoder on all conditions, then, for each rare condition, fine-tunes a copy on that condition plus the ~10% of conditions that a **condition knowledge graph** marks as most similar, with **joint adversarial domain adaptation** so the borrowed patients look like the target ones.

- **Task** (`KnowRareICUMortalityMIMIC4`): death in the ICU from the first 24 hourly bins of the stay (48 labs and vitals, plus age, sex, height and weight). Every stay is labelled with its condition: the 3-character ICD category of the admission's primary diagnosis. This ports the paper's eICU ICU-mortality task to MIMIC-IV.
- **Model** (`KnowRare` and its helpers in `pyhealth.models.knowrare`): knowledge graph, TuckER embedding, source-condition selection, next-step pre-training, adversarial fine-tuning.
- **Baselines** already in PyHealth, as in the paper's Table 2: LSTM (`RNN`), `Transformer` and `RETAIN` trained on all conditions, and an LSTM trained on the rare condition alone. Two ablations: KnowRare without the adversarial term and without pre-training.
- **Metrics**: AUPRC and AUROC on the test patients of the rare conditions, pooled over conditions (as the authors' evaluation code does).

`DATA = "demo"` runs end to end on the open **MIMIC-IV Clinical Database Demo** (100 patients, no credentials) in about a minute on a CPU. **That run is a smoke test only:** the demo has about 100 eligible ICU stays and 10 deaths, no condition reaches the paper's ten-patient minimum, and the rare-condition test sets hold only a couple of deaths in total, so its numbers say nothing about performance. `DATA = "full"` runs the paper's protocol on credentialed MIMIC-IV.

## Setup

KnowRare will be on PyHealth's master branch once its pull request is merged, and in the next PyPI release after that. Until then, install from GitHub (the cell installs only on Colab; locally it uses the PyHealth you have).

In [1]:
import sys

if "google.colab" in sys.modules:
    !pip install -q "git+https://github.com/sunlabuiuc/PyHealth"

In [2]:
import copy
import logging
import os
import urllib.request
import warnings

os.environ["TQDM_DISABLE"] = "1"  # keep the notebook output short
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

from pyhealth.datasets import MIMIC4EHRDataset, PatientSplit, get_dataloader
from pyhealth.models import RETAIN, RNN, KnowRare, Transformer
from pyhealth.models.knowrare import (
    ConditionKnowledgeGraph,
    TuckER,
    select_source_conditions,
)
from pyhealth.tasks import KnowRareICUMortalityMIMIC4
from pyhealth.tasks.knowrare_mimic4 import select_rare_conditions
from pyhealth.trainer import Trainer

logging.getLogger("pyhealth").setLevel(logging.ERROR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Configuration

`DATA = "demo"` downloads the MIMIC-IV demo (about 15 MB). For `DATA = "full"`, point `MIMIC4_ROOT` at your MIMIC-IV copy (the folder with `hosp/` and `icu/`, e.g. on a mounted Google Drive) and use a GPU runtime. Building the event table reads `chartevents` once and caches it in `CACHE_DIR`.

The hyperparameters are the authors' tuned values for ICU mortality on eICU (`utils/best_hparams.py` in their repository) and the paper's training settings: hidden size 128, at most 100 epochs with early stopping after 10, 10% of the conditions as sources, five seeds. The knowledge graph keeps the top 60% of edges, the authors' choice for the single-centre MIMIC-III (MIMIC-IV is single-centre too; they use 5% on the multi-centre eICU). The demo run shortens training.

In [3]:
DATA = "demo"  # "demo" or "full"
MIMIC4_ROOT = "/content/drive/MyDrive/mimic-iv/2.2"  # used when DATA == "full"

TABLES = ["labevents", "chartevents", "outputevents", "transfers",
          "diagnoses_icd", "prescriptions"]
CONFIG = {
    "seeds": [0, 1, 2, 3, 4],
    "split": (0.67, 0.16, 0.17),     # train / val / test, by patient
    "hidden_dim": 128,
    "pretrain_lr": 5.4e-4, "pretrain_batch": 32,
    "lr": 6.7e-4, "batch": 64,
    "adversarial_weight": 0.02,      # lambda in Eq. 12
    "source_ratio": 0.1,             # k = int(0.1 * #conditions) + 1
    "top_edge_ratio": 0.6,
    "tucker_dim": 64, "tucker_lr": 4.4e-4, "tucker_batch": 32, "tucker_epochs": 300,
    "epochs": 100, "patience": 10,
}
if DATA == "demo":
    CONFIG.update(epochs=30, tucker_epochs=200)

if DATA == "demo":
    ROOT, CACHE_DIR = "mimic-iv-demo-2.2", "cache-knowrare-demo"
    base_url = "https://physionet.org/files/mimic-iv-demo/2.2/"
    files = ["hosp/admissions", "hosp/patients", "hosp/labevents", "hosp/d_labitems",
             "hosp/transfers", "hosp/diagnoses_icd", "hosp/prescriptions",
             "icu/icustays", "icu/chartevents", "icu/outputevents"]
    for name in files:
        path = os.path.join(ROOT, name + ".csv.gz")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            urllib.request.urlretrieve(base_url + name + ".csv.gz", path)
    print("demo tables ready:", len(files))
else:
    ROOT, CACHE_DIR = MIMIC4_ROOT, "cache-knowrare-mimic4"
    assert os.path.isdir(ROOT), "set MIMIC4_ROOT to your MIMIC-IV folder"

demo tables ready: 10


## Data and rare conditions

`set_task(..., split=PatientSplit(...))` splits patients 67/16/17 as in the paper and fits the processors (the z-scoring and the training-mean imputation) on the training patients only.

Rare conditions are chosen with the paper's rule (`select_rare_conditions`): fewer than one patient in 2,000, at least ten patients, at least one death, least prevalent first, ten conditions. They must also appear in the training and test splits (the paper stratifies its split by condition; `PatientSplit` does not). On the demo no condition has ten patients, so every condition present in both the training and test splits is used instead.

In [4]:
base = MIMIC4EHRDataset(root=ROOT, tables=TABLES, cache_dir=CACHE_DIR)
task = KnowRareICUMortalityMIMIC4()


def label(ds):
    return np.array([float(s["mortality"]) for s in ds])


def load_split(seed):
    train, val, test = base.set_task(
        task, split=PatientSplit(ratios=CONFIG["split"], seed=seed), num_workers=1)
    conds = [[s["condition"] for s in part] for part in (train, val, test)]
    if DATA == "demo":
        targets = sorted(set(conds[0]) & set(conds[2]))
    else:
        parts = (train, val, test)
        targets = select_rare_conditions(
            sum(conds, []), np.concatenate([label(p) for p in parts]),
            patient_ids=[s["patient_id"] for p in parts for s in p],
            required=[conds[0], conds[2]])
    return (train, val, test), conds, targets


(train, val, test), conds, targets = load_split(CONFIG["seeds"][0])
for name, part in zip(("train", "val", "test"), (train, val, test)):
    y = label(part)
    print(f"{name:5s}: {len(part):5d} stays, {int(y.sum()):4d} deaths ({y.mean():.1%}), "
          f"{len(set(s['patient_id'] for s in part))} patients")
print(f"{len(set(conds[0]))} conditions in training; rare targets (seed 0): {targets}")
sample = train[0]
print({k: tuple(v.shape) if hasattr(v, "shape") else v for k, v in sample.items()
       if k in ("dynamic", "static", "mortality", "condition")})

train:    68 stays,    6 deaths (8.8%), 48 patients
val  :    14 stays,    2 deaths (14.3%), 11 patients
test :    14 stays,    2 deaths (14.3%), 13 patients
41 conditions in training; rare targets (seed 0): ['ICD10:I25', 'ICD10:I26', 'ICD10:I82', 'ICD9:531']
{'dynamic': (24, 48), 'static': (8,), 'mortality': (1,), 'condition': 'ICD9:788'}


## Step 1: condition knowledge graph and source selection

`ConditionKnowledgeGraph.from_samples` builds the three relations of the paper from the training stays only: diagnosis co-occurrence (Eq. 1), similarity of the conditions' record statistics (Eqs. 2-3, top 50% kept) and Jaccard similarity of their drugs (Eq. 4, top 50% kept). `TuckER` embeds the graph (Eq. 5) and `select_source_conditions` takes the `k` conditions with the most similar embeddings (Eq. 6).

In [5]:
def condition_graph(train, seed):
    graph = ConditionKnowledgeGraph.from_samples(train)
    torch.manual_seed(seed)
    tucker = TuckER(len(graph.conditions), len(graph.relations),
                    CONFIG["tucker_dim"], CONFIG["tucker_dim"]).to(device)
    losses = tucker.fit(graph.triples(CONFIG["top_edge_ratio"]), epochs=CONFIG["tucker_epochs"],
                        lr=CONFIG["tucker_lr"], batch_size=CONFIG["tucker_batch"], seed=seed)
    k = int(CONFIG["source_ratio"] * len(graph.conditions)) + 1
    return graph, tucker.entity_embeddings.cpu(), k, losses


graph, embeddings, k, losses = condition_graph(train, CONFIG["seeds"][0])
edges = {r: int((w > 0).sum()) for r, w in graph.weights.items()}
print(f"{len(graph.conditions)} nodes, edges per relation: {edges}, k = {k}")
print(f"TuckER loss {losses[0]:.3f} -> {losses[-1]:.3f}")
target = targets[0]
print(f"sources for {target}:", select_source_conditions(embeddings, graph.conditions, target, k))

41 nodes, edges per relation: {'diagnosis': 194, 'record': 820, 'drug': 842}, k = 5
TuckER loss 0.906 -> 0.403
sources for ICD10:I25: ['ICD10:K26', 'ICD10:S12', 'ICD9:821', 'ICD10:T81', 'ICD9:157']


## Step 2: condition-agnostic pre-training

`KnowRare.pretrain` trains the encoder (one-layer LSTM over the 24 hourly bins, two-layer MLP over the static variables) and a decoder to predict the next hour from the hours so far (Eq. 8), on the training stays of all conditions, keeping the epoch with the lowest validation error.

In [6]:
def loader(ds, shuffle=False, batch=None):
    return get_dataloader(ds, batch or CONFIG["batch"], shuffle=shuffle, seed=0)


def pretrained_model(train, val, seed):
    torch.manual_seed(seed)
    model = KnowRare(train, hidden_dim=CONFIG["hidden_dim"],
                     adversarial_weight=CONFIG["adversarial_weight"]).to(device)
    history = model.pretrain(loader(train, True, CONFIG["pretrain_batch"]),
                             loader(val, batch=CONFIG["pretrain_batch"]),
                             epochs=CONFIG["epochs"], lr=CONFIG["pretrain_lr"],
                             patience=CONFIG["patience"])
    return model, history


pretrained, history = pretrained_model(train, val, CONFIG["seeds"][0])
print(f"{len(history['train'])} epochs, train MSE {history['train'][0]:.3f} -> "
      f"{history['train'][-1]:.3f}, best val MSE {min(history['val']):.3f}")

30 epochs, train MSE 0.674 -> 0.310, best val MSE 0.994


## Step 3: adaptation to each rare condition, and baselines

For each target, a copy of the pre-trained model is fine-tuned with PyHealth's `Trainer` on the training stays of the target and its `k` source conditions. `set_domains` adds the discriminator over the `k + 1` conditions (fed the latent representation and the predicted risk, Eq. 9) and the inverse-prevalence sample weights; the gradient reversal layer makes one optimizer step the minimax update of Eq. 10. Model selection uses validation AUPRC on the same conditions (validation loss when they have a single class, as can happen on the demo).

The baselines are trained on all training stays ("all") or on the target's stays alone ("single"). Every model scores the test stays of every target, and the predictions are pooled over targets.

In [7]:
def fit(model, train_ds, val_ds):
    y_val = label(val_ds) if val_ds is not None else np.zeros(0)
    if len(y_val) and 0 < y_val.sum() < len(y_val):
        monitor, criterion = "pr_auc", "max"
    elif len(y_val):
        monitor, criterion = "loss", "min"
    else:
        monitor, criterion = None, "max"
    trainer = Trainer(model, metrics=["pr_auc", "roc_auc"], device=device,
                      enable_logging=False)
    trainer.train(loader(train_ds, True), loader(val_ds) if monitor else None,
                  epochs=CONFIG["epochs"], optimizer_params={"lr": CONFIG["lr"]},
                  monitor=monitor, monitor_criterion=criterion, patience=CONFIG["patience"])
    return model


@torch.no_grad()
def predict(model, ds):
    model.eval()
    return np.concatenate([model(**b)["y_prob"][:, 0].cpu().numpy() for b in loader(ds)])


def subset(ds, conditions, keep):
    idx = [i for i, c in enumerate(conditions) if c in keep]
    return ds.subset(idx) if idx else None


def run_seed(seed):
    (train, val, test), conds, targets = load_split(seed)
    graph, embeddings, k, _ = condition_graph(train, seed)
    pretrained, _ = pretrained_model(train, val, seed)
    test_sets = {t: subset(test, conds[2], {t}) for t in targets}
    preds = {"y": np.concatenate([label(test_sets[t]) for t in targets])}

    baselines = {"LSTM (all)": lambda d: RNN(d, rnn_type="LSTM"),
                 "Transformer (all)": Transformer, "RETAIN (all)": RETAIN}
    for name, make in baselines.items():
        torch.manual_seed(seed)
        model = fit(make(train).to(device), train, val)
        preds[name] = np.concatenate([predict(model, test_sets[t]) for t in targets])
        if name == "LSTM (all)":
            sanity = {"train": (label(train), predict(model, train)),
                      "test": (label(test), predict(model, test))}

    variants = {"LSTM (single)": None, "KnowRare": "full",
                "w/o adversarial adaptation": "no_adv", "w/o pre-training": "no_pre"}
    for name in variants:
        preds[name] = []
    for t in targets:
        sources = select_source_conditions(embeddings, graph.conditions, t, k)
        domains = {t, *sources}
        train_t = subset(train, conds[0], domains)
        val_t = subset(val, conds[1], domains)
        for name, variant in variants.items():
            torch.manual_seed(seed)
            if variant is None:  # the target's own stays only
                model = RNN(train, rnn_type="LSTM").to(device)
                fit(model, subset(train, conds[0], {t}), subset(val, conds[1], {t}))
            else:
                if variant == "no_pre":
                    model = KnowRare(train, hidden_dim=CONFIG["hidden_dim"]).to(device)
                else:
                    model = copy.deepcopy(pretrained)
                model.adversarial_weight = (
                    0.0 if variant == "no_adv" else CONFIG["adversarial_weight"])
                model.set_domains(t, sources, conditions=[s["condition"] for s in train_t])
                fit(model, train_t, val_t)
            preds[name].append(predict(model, test_sets[t]))
    for name in variants:
        preds[name] = np.concatenate(preds[name])
    return preds, sanity, targets


runs = []
for seed in CONFIG["seeds"]:
    preds, sanity, seed_targets = run_seed(seed)
    runs.append((preds, sanity))
    print(f"seed {seed}: {len(seed_targets)} targets, {len(preds['y'])} test stays, "
          f"{int(preds['y'].sum())} deaths")

seed 0: 4 targets, 5 test stays, 0 deaths


seed 1: 7 targets, 10 test stays, 0 deaths


seed 2: 4 targets, 8 test stays, 1 deaths


seed 3: 9 targets, 10 test stays, 1 deaths


seed 4: 4 targets, 5 test stays, 0 deaths


## Results

In [8]:
def scores(y, p):
    if len(y) == 0 or y.min() == y.max():
        return np.nan, np.nan  # undefined with a single class
    return average_precision_score(y, p), roc_auc_score(y, p)


models = [m for m in runs[0][0] if m != "y"]
rows = []
for m in models:
    per_seed = np.array([scores(p["y"], p[m]) for p, _ in runs])
    y_all = np.concatenate([p["y"] for p, _ in runs])
    pooled = scores(y_all, np.concatenate([p[m] for p, _ in runs]))
    rows.append({
        "model": m,
        "AUPRC mean (std) over seeds": f"{np.nanmean(per_seed[:, 0]):.3f} ({np.nanstd(per_seed[:, 0]):.3f})",
        "AUROC mean (std) over seeds": f"{np.nanmean(per_seed[:, 1]):.3f} ({np.nanstd(per_seed[:, 1]):.3f})",
        "seeds with both classes": int(np.isfinite(per_seed[:, 0]).sum()),
        "AUPRC pooled": round(pooled[0], 3), "AUROC pooled": round(pooled[1], 3),
    })
results = pd.DataFrame(rows).set_index("model")
y_all = np.concatenate([p["y"] for p, _ in runs])
print(f"pooled over seeds: {len(y_all)} rare-condition test stays, {int(y_all.sum())} deaths "
      f"(prevalence {y_all.mean():.1%}, the AUPRC of a random ranking)")
results

pooled over seeds: 38 rare-condition test stays, 2 deaths (prevalence 5.3%, the AUPRC of a random ranking)


,AUPRC mean (std) over seeds,AUROC mean (std) over seeds,seeds with both classes,AUPRC pooled,AUROC pooled
model,,,,,
LSTM (all),0.217 (0.117),0.357 (0.357),2,0.129,0.653
Transformer (all),0.417 (0.083),0.802 (0.087),2,0.078,0.514
RETAIN (all),0.150 (0.050),0.214 (0.214),2,0.063,0.333
LSTM (single),0.146 (0.021),0.254 (0.032),2,0.049,0.208
KnowRare,0.292 (0.042),0.690 (0.024),2,0.143,0.750
w/o adversarial adaptation,0.267 (0.067),0.635 (0.079),2,0.134,0.722
w/o pre-training,0.208 (0.042),0.476 (0.190),2,0.097,0.625


### Sanity checks

Near-chance scores need an explanation before they are written up. The checks below use all test stays, not only the rare conditions: the label balance, and the AUROC on the training stays versus all test stays of the LSTM (all) baseline and of a logistic regression on the 24-hour mean of each variable plus the static variables (mean over seeds).

In [9]:
def flat(ds):
    return np.stack([np.concatenate([s["dynamic"].numpy().mean(0), s["static"].numpy()])
                     for s in ds])


check = {"LSTM (all), training stays": [s["train"] for _, s in runs],
         "LSTM (all), all test stays": [s["test"] for _, s in runs],
         "logistic regression, training stays": [],
         "logistic regression, all test stays": []}
for seed in CONFIG["seeds"]:
    (tr, _, te), _, _ = load_split(seed)
    clf = LogisticRegression(max_iter=2000, C=0.1).fit(flat(tr), label(tr))
    check["logistic regression, training stays"].append((label(tr), clf.predict_proba(flat(tr))[:, 1]))
    check["logistic regression, all test stays"].append((label(te), clf.predict_proba(flat(te))[:, 1]))
# Scores per seed, then the mean over seeds (scores of different models are not pooled).
check = {name: np.nanmean([scores(y, p) for y, p in pairs], axis=0)
         for name, pairs in check.items()}
train_y = np.concatenate([s["train"][0] for _, s in runs])
print(f"deaths in training splits: {int(train_y.sum())} of {len(train_y)} stays over "
      f"{len(CONFIG['seeds'])} seeds ({train_y.mean():.1%})")
pd.DataFrame(check, index=["AUPRC", "AUROC"]).T.round(3)

deaths in training splits: 36 of 320 stays over 5 seeds (11.2%)


,AUPRC,AUROC
"LSTM (all), training stays",0.224,0.551
"LSTM (all), all test stays",0.336,0.665
"logistic regression, training stays",0.864,0.972
"logistic regression, all test stays",0.295,0.478


## Comparison with the paper

The paper's Table 2 (mean (std) over five seeds; ICU mortality is on eICU, ten rare conditions):

| Method | ICU mortality AUPRC | ICU mortality AUROC | 90-day mortality AUPRC (MIMIC-III) | 90-day mortality AUROC (MIMIC-III) |
|---|---|---|---|---|
| LSTM (single) | 0.427 (0.113) | 0.491 (0.106) | 0.339 (0.039) | 0.432 (0.072) |
| LSTM (all) | 0.569 (0.039) | 0.725 (0.051) | 0.725 (0.028) | 0.789 (0.022) |
| Transformer (all) | 0.636 (0.013) | 0.770 (0.023) | 0.665 (0.045) | 0.751 (0.048) |
| RETAIN (all) | 0.532 (0.046) | 0.733 (0.025) | 0.679 (0.058) | 0.770 (0.032) |
| Stable-CRP (best domain adaptation baseline) | 0.631 (0.053) | 0.691 (0.027) | 0.726 (0.041) | 0.819 (0.027) |
| **KnowRare** | **0.709 (0.022)** | 0.757 (0.012) | **0.744 (0.051)** | 0.797 (0.032) |
| KnowRare w/o domain adaptation (Table 4) | 0.713 (0.025) | 0.725 (0.013) | 0.679 (0.061) | 0.785 |
| KnowRare w/o pre-training (Table 4) | 0.628 (0.049) | 0.693 (0.009) | 0.640 (0.090) | 0.730 (0.054) |

On eICU, APACHE IV reaches 0.639 AUPRC and 0.701 AUROC on the same rare conditions (Table 3).

**How to read the demo numbers above.** They are a smoke test, not a result, and no ranking of the models should be read from them. The demo has 96 eligible ICU stays and 10 deaths. Over the five seeds, the rare-condition test sets hold 38 stays and 2 deaths, one in each of two seeds; the other three seeds have no death, so their scores are undefined. Each per-seed AUROC is therefore the rank of a single death among 8 to 10 stays, and the pooled columns mix scores from different models. The sanity checks show that the demo has too few deaths to learn from: the logistic regression fits its training stays (AUROC about 0.97) but is at chance on the test stays (about 0.48), and the LSTM (all) baseline barely fits its training stays (about 0.55) because early stopping on a validation split of about 14 stays with 2 deaths keeps near-initial weights. Its test AUROC over all test stays (about 0.67, also with about 2 deaths per seed) is noise as well. Run `DATA = "full"` for numbers comparable to the paper.

**What differs from the paper**, so full-data numbers are comparable in trend rather than identical:

- **Data**: MIMIC-IV instead of eICU (single centre, one hospital). The cohort is YAIB's (adults, ICU stays of at least 30 hours, deaths in the first 30 hours excluded) instead of ICU stays of at least 24 hours; the 48 YAIB variables replace Supplementary Table S1, and height and weight replace race. ICD-9 and ICD-10 categories are kept side by side, so a condition coded in both versions counts as two conditions. Prevalence is computed over the task's cohort.
- **Split**: patients are split 67/16/17 without stratifying by condition, and targets must appear in the training and test splits.
- **Training**: one optimizer with a gradient reversal layer instead of alternating discriminator and encoder updates (same objective); no learning-rate warm-up or decay; TuckER is trained for a fixed number of epochs without the authors' held-out-edge early stopping.
- **Metric**: AUPRC is average precision (scikit-learn), the authors use the trapezoidal area under the precision-recall curve.

## What to check (for the KnowRare authors)

1. The task in `pyhealth/tasks/knowrare_mimic4.py`: primary condition = 3-character category of the `seq_num` 1 diagnosis; forward then backward fill within the stay, then the training mean, then z-scores with training statistics.
2. The knowledge graph in `pyhealth/models/knowrare.py` (`ConditionKnowledgeGraph`): it follows Eqs. 1-4 of the paper. Your notebooks differ in places (min-max scaled co-occurrence counts instead of row normalization; record statistics that include the missing-value rate and a min-max scaled distance); which should PyHealth follow?
3. The adaptation step (`KnowRare.forward` and `set_domains`): the discriminator classifies the `k + 1` selected conditions from `[h_T, y_prob]` and the inverse-prevalence weights multiply the loss, as written in the paper. Your code classifies all training conditions from `[h_T, linear(logits)]` and resamples batches instead.
4. Model selection: here on validation stays of the target and its sources; your code validates on the top-10% neighbours.